# PCC_LEAKAGE_FREE_RERUN_2026 smoke
Pinned one-fold, one-held-out-case execution.

In [ ]:
from pathlib import Path
import csv, hashlib, json, shutil, subprocess, sys
import numpy as np
repo = Path('/kaggle/working/PCC')
subprocess.run(['git', 'clone', 'https://github.com/changxinjiresearch/PCC.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', '0f515597d73367f73f6f0c913f4fa3e0ebbd963c'], check=True)
subprocess.run([sys.executable, str(repo / 'experiments/run_pcc_leakage_free_full.py'), '--config', str(repo / 'configs/pcc_leakage_free_canonical.yaml'), '--smoke'], cwd=repo, check=True)
root = Path('/kaggle/working/PCC_LEAKAGE_FREE_RERUN_2026')
case = next((root / 'held_out_p0').iterdir())
retro = root / 'retrospective' / 'cases' / case.name
paths = [case / 'P0_float32.npy'] + [retro / f'P{i}.npy' for i in range(1, 11)] + [retro / 'pcc_correction.npy']
arrays = [np.load(path) for path in paths]
assert all(array.dtype == np.float32 and np.isfinite(array).all() and array.shape == arrays[0].shape for array in arrays)
assert np.array_equal(arrays[-2], arrays[-1])
with (retro / 'method_metrics.csv').open() as stream: methods = list(csv.DictReader(stream))
with (retro / 'pcc_round_trajectory.csv').open() as stream: rounds = list(csv.DictReader(stream))
assert len(methods) == 7 and len(rounds) == 10
summary = {'status':'PASS','case_id':case.name,'shape':list(arrays[0].shape),'p0_dtype':str(arrays[0].dtype),'p0_finite':bool(np.isfinite(arrays[0]).all()),'methods':[row['method'] for row in methods],'rounds':[row['round'] for row in rounds],'p10_equals_final':bool(np.array_equal(arrays[-2], arrays[-1])),'hashes':{path.name:hashlib.sha256(path.read_bytes()).hexdigest() for path in paths},'case_manifest_sha256':hashlib.sha256((root/'LOCKED_CASE_MANIFEST.csv').read_bytes()).hexdigest(),'fold_manifest_sha256':hashlib.sha256((root/'LOCKED_FOLD_MANIFEST.csv').read_bytes()).hexdigest()}
evidence = repo / 'smoke_validation'; evidence.mkdir(exist_ok=True)
(evidence / 'smoke_validation.json').write_text(json.dumps(summary, indent=2))
for path in [root/'LOCKED_CASE_MANIFEST.csv', root/'LOCKED_FOLD_MANIFEST.csv', retro/'method_metrics.csv', retro/'pcc_round_trajectory.csv', case/'P0_COMPLETE.json', retro/'RETROSPECTIVE_COMPLETE.json']: shutil.copy2(path, evidence/path.name)
print(json.dumps(summary, indent=2))
